<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-08-gradio-rag-chatbot/notebooks/Pfizer_Capstone_Full_RAG_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Full RAG UI — capstone build
**Pfizer AI Document Intelligence Externship**

End-to-end retrieval-augmented generation over pharmaceutical supplier documents, with a
Gradio front end built to be demoed rather than debugged.

**Pipeline:** PDF (digital *or* scanned) → text extraction with OCR fallback → per-page
metadata tagging → overlapping chunks → sentence-transformers embeddings → FAISS index →
metadata-filtered retrieval → grounded prompt → open-source LLM → answer + sources +
confidence + chunk count.

| component | choice |
|---|---|
| extraction | PyMuPDF, with Tesseract OCR when a page has no text layer |
| metadata | `doc_type`, `doc_id`, `page`, `page_range`, `source_file`, `chunk_index`, `ocr` |
| embeddings | `BAAI/bge-small-en-v1.5` (sentence-transformers) |
| index | FAISS `IndexFlatIP` over L2-normalised vectors (exact cosine) |
| generation | Mistral-7B-Instruct-v0.3 in 4-bit on GPU, Qwen2.5-1.5B-Instruct on CPU |
| UI | Gradio Blocks: packet summary, document inventory, chat, and an evidence panel with confidence and cited passages |

No API keys and no Gemini. A GPU makes it faster and sharper; without one the notebook
still runs end to end.

## 0. Install

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install gradio pymupdf sentence-transformers faiss-cpu pytesseract pillow gdown \
                "transformers>=4.44" accelerate bitsandbytes

import io, os, re, json, datetime
import numpy as np
import torch, pymupdf, faiss, pytesseract, gradio as gr
from PIL import Image
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("gradio", gr.__version__, "| faiss ok | CUDA", torch.cuda.is_available())

gradio 6.26.0 | faiss ok | CUDA False


## 1. Document processing

Each page is read with PyMuPDF first. If a page comes back with almost no text it is a
scanned image, so it gets rasterised at 200 dpi and passed through Tesseract instead. Pages
record which route they took, because an OCR page is noisier and that is worth surfacing.

In [ ]:
MIN_CHARS = 40          # below this a page is treated as scanned
OCR_DPI  = 200

def read_page(page):
    """Return (text, used_ocr). Falls back to Tesseract when there is no text layer."""
    text = page.get_text("text").strip()
    if len(text) >= MIN_CHARS:
        return re.sub(r"\n{3,}", "\n\n", text), False
    try:
        pix = page.get_pixmap(dpi=OCR_DPI)
        img = Image.open(io.BytesIO(pix.tobytes("png")))
        ocr = pytesseract.image_to_string(img).strip()
        return re.sub(r"\n{3,}", "\n\n", ocr), True
    except Exception as e:
        print(f"  [warn] OCR failed on a page ({type(e).__name__})")
        return text, False

### Metadata tagging

`doc_type` comes from the document title in the page header. A `(continued)` page inherits
the previous page's type, and two same-titled pages carrying different lot numbers are two
different documents, so `doc_id` counts real documents rather than page runs. Every document
also records the page range it spans.

In [ ]:
DOC_TYPES = ["Cover Letter", "Certificate of Quality", "Packaging Specification",
             "BSE/TSE Declaration", "Material Description Sheet",
             "Supplier Qualification Record", "Chain of Custody", "Unknown"]
SLUG = {t: re.sub(r"[^a-z]+", "_", t.lower()).strip("_") for t in DOC_TYPES}

HEADING_CUES = {
    "certificate of quality": "Certificate of Quality",
    "quality certificate": "Certificate of Quality",
    "packaging component specification": "Packaging Specification",
    "declaration regarding transmissible spongiform": "BSE/TSE Declaration",
    "bse/tse": "BSE/TSE Declaration",
    "animal origin statement": "BSE/TSE Declaration",
    "material description sheet": "Material Description Sheet",
    "compliance bulletin": "Material Description Sheet",
    "supplier qualification record": "Supplier Qualification Record",
    "vendor qualification record": "Supplier Qualification Record",
    "chain of custody": "Chain of Custody",
    "certificate of processing": "Chain of Custody",
    "to whom it may concern": "Cover Letter",
    "dear ": "Cover Letter",
    "subject:": "Cover Letter",
}
CONTINUATION = ("(continued)", "cont'd", "page 2 of", "page 3 of")
ID_PATTERNS = (r"lot number\s*:?\s*([a-z0-9\-]{4,})",
               r"document number\s*:?\s*([a-z0-9\-/]{4,})",
               r"product article number\s*:?\s*([0-9 ]{6,})")

def identifiers(text):
    low, ids = text.lower(), set()
    for p in ID_PATTERNS:
        ids.update(m.group(1).strip() for m in re.finditer(p, low))
    return ids

def classify_page(text):
    head = "\n".join(text.strip().splitlines()[:12]).lower()   # letterheads push the title down
    for k, v in HEADING_CUES.items():
        if k in head:
            return v
    low = text.lower()
    for k, v in HEADING_CUES.items():
        if k in low:
            return v
    return "Unknown"


def tag_pages(path, filename):
    """One tagged record per page, including the page range of its parent document."""
    doc = pymupdf.open(path)
    pages, prev_type, prev_text, doc_id, counters = [], None, None, None, {}
    for i, pg in enumerate(doc):
        text, used_ocr = read_page(pg)
        dtype = classify_page(text)
        head = "\n".join(text.strip().splitlines()[:4]).lower()
        is_cont = any(c in head for c in CONTINUATION) and prev_type is not None
        if is_cont:
            dtype = prev_type
        new_doc = (not is_cont) and (
            prev_type is None or dtype != prev_type
            or (identifiers(text) and identifiers(prev_text)
                and not (identifiers(text) & identifiers(prev_text))))
        if new_doc:
            s = SLUG.get(dtype, "unknown")
            counters[s] = counters.get(s, 0) + 1
            doc_id = f"{s}_{counters[s]:02d}"
        pages.append({"page": i, "doc_type": dtype, "doc_id": doc_id,
                      "source_file": filename, "text": text, "ocr": used_ocr})
        prev_type, prev_text = dtype, text

    # page range per document, so a chunk can say "pages 3-4" not just "page 3"
    spans = {}
    for p in pages:
        lo, hi = spans.get(p["doc_id"], (p["page"], p["page"]))
        spans[p["doc_id"]] = (min(lo, p["page"]), max(hi, p["page"]))
    for p in pages:
        lo, hi = spans[p["doc_id"]]
        p["page_range"] = f"{lo+1}" if lo == hi else f"{lo+1}-{hi+1}"
    return pages


CHUNK, OVERLAP = 700, 120
SEPARATORS = ["\n\n", "\n", ". ", " "]        # try to break on the largest natural boundary

def split_text(t, size=CHUNK, overlap=OVERLAP):
    """Recursive-style split: never cut a word or a phrase in half if a boundary is near."""
    out, start = [], 0
    while start < len(t):
        end = min(start + size, len(t))
        if end < len(t):
            for sep in SEPARATORS:                       # look for a boundary in the back half
                p = t.rfind(sep, start + size // 2, end)
                if p != -1:
                    end = p + len(sep)
                    break
        piece = t[start:end].strip()
        if piece:
            out.append(piece)
        if end >= len(t):
            break
        start = max(end - overlap, start + 1)
    return out


def chunk_pages(pages):
    """Overlapping chunks that inherit the full metadata of their page."""
    out = []
    for p in pages:
        if len(p["text"]) < 20:
            continue
        for piece in split_text(p["text"]):
            out.append({**{k: v for k, v in p.items() if k != "text"},
                        "text": piece, "chunk_index": len(out)})
    return out

## 2. Embeddings and the FAISS index

Vectors are L2-normalised and stored in a FAISS `IndexFlatIP`, so an inner product search is
an exact cosine search. Filtering by `doc_type` happens by over-fetching from FAISS and then
keeping only the chunks whose metadata matches, which keeps one index for the whole corpus.

In [ ]:
EMBED_ID = "BAAI/bge-small-en-v1.5"
embedder = SentenceTransformer(EMBED_ID, device="cuda" if torch.cuda.is_available() else "cpu")
DIM = embedder.get_sentence_embedding_dimension()

def build_index(chunks):
    vecs = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                           batch_size=32, show_progress_bar=False).astype("float32")
    index = faiss.IndexFlatIP(DIM)     # inner product on unit vectors == cosine
    index.add(vecs)
    return index


def search(question, store, doc_type="All types", k=4):
    """Retrieve k chunks, optionally restricted to one document type."""
    q = embedder.encode([question], normalize_embeddings=True).astype("float32")
    over = k * 6 if doc_type != "All types" else k        # over-fetch so the filter has room
    scores, ids = store["index"].search(q, min(over, len(store["chunks"])))
    hits = []
    for score, i in zip(scores[0], ids[0]):
        if i < 0:
            continue
        c = store["chunks"][i]
        if doc_type != "All types" and c["doc_type"] != doc_type:
            continue
        hits.append({**c, "score": float(score)})
        if len(hits) == k:
            break
    return hits

print("embedder ready, dim", DIM)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedder ready, dim 384


/tmp/ipykernel_3272/75530812.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = embedder.get_sentence_embedding_dimension()


## 3. The open-source model

In [ ]:
GPU_MODEL = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"   # 4-bit Mistral-7B-Instruct-v0.3
CPU_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"                  # fallback when no GPU is free

USE_GPU  = torch.cuda.is_available()
MODEL_ID = GPU_MODEL if USE_GPU else CPU_MODEL
MAX_NEW  = 320 if USE_GPU else 180

# Load once. Re-running this cell without the guard loads a second copy and, on a CPU
# runtime with 12 GB, that is enough to crash the session.
if "model" in globals() and globals().get("_loaded_id") == MODEL_ID:
    print("model already loaded, reusing:", MODEL_ID)
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if USE_GPU:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16,
                                 bnb_4bit_use_double_quant=True)
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                                     device_map="auto", torch_dtype=torch.float16)
    else:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
    model.eval()
    _loaded_id = MODEL_ID
    print(f"generation on {'GPU' if USE_GPU else 'CPU'} with {MODEL_ID}")

model already loaded, reusing: Qwen/Qwen2.5-1.5B-Instruct


## 4. Prompting and the RAG pipeline

Passages are numbered and labelled with their file, page and document type, and the model is
told to cite those numbers and to refuse when the passages do not hold the answer. Confidence
is reported from the retrieval scores rather than the model's tone, because a model's
certainty is not evidence and a similarity score at least is.

In [ ]:
STYLE = {
    "Short answer":    "Answer in one or two sentences.",
    "Detailed answer": "Answer in a short paragraph, quoting exact figures, dates and identifiers.",
    "Bullet summary":  "Answer as 3 to 5 short bullet points, one fact per bullet.",
}

def build_prompt(question, hits, mode):
    context = "\n\n".join(
        f"[{i+1}] {h['doc_type']} — {h['source_file']}, page {h['page']+1}\n{h['text']}"
        for i, h in enumerate(hits))
    return (
        "You are a document analyst answering questions about pharmaceutical supplier documents.\n"
        "Use ONLY the numbered passages below. Cite the passage numbers you used, like [1] or [2].\n"
        "If the passages do not contain the answer, say so plainly instead of guessing.\n"
        "Answer the question directly and make your answer agree with the passages: if a passage\n"
        "states that something is absent or not present, do not report it as present.\n"
        f"{STYLE.get(mode, STYLE['Short answer'])}\n\n"
        f"PASSAGES:\n{context}\n\nQUESTION: {question}\n\nANSWER:")


@torch.inference_mode()
def generate(prompt):
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                         tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                         temperature=None, top_p=None, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def confidence(hits):
    """Top similarity, plus how much of the top-k agrees with it. Returns (pct, label)."""
    if not hits:
        return 0.0, "none"
    top = hits[0]["score"]
    support = sum(1 for h in hits if h["score"] >= top - 0.08) / len(hits)
    pct = round(100 * (0.75 * top + 0.25 * support), 1)
    label = "High" if top >= 0.62 else "Medium" if top >= 0.48 else "Low"
    return pct, label


def rag_answer(question, store, doc_type="All types", mode="Short answer", k=4):
    """Retrieval -> context -> prompt -> model -> answer + sources + confidence."""
    hits = search(question, store, doc_type, k)
    if not hits:
        return {"answer": "Nothing in the indexed documents matched that question.",
                "sources": [], "confidence": 0.0, "confidence_label": "none", "chunks_used": 0}
    answer = generate(build_prompt(question, hits, mode))
    pct, label = confidence(hits)
    sources = [{"n": i + 1, "doc_type": h["doc_type"], "doc_id": h["doc_id"],
                "file": h["source_file"], "page": h["page"] + 1,
                "page_range": h["page_range"], "ocr": h["ocr"],
                "score": round(h["score"], 3), "text": h["text"]}
               for i, h in enumerate(hits)]
    return {"answer": answer, "sources": sources, "confidence": pct,
            "confidence_label": label, "chunks_used": len(hits)}

## 5. Loading documents

A sample pharmaceutical packet is bundled so the app can be demoed without an upload.

In [ ]:
import gdown
SAMPLES = [("pharma-blob-sample.pdf", "1fJ7DszcjB_TerDTAhXfAw74VqJBElTyw"),
           ("pharma-blob-test.pdf",   "1atOj_CzvPRm-CskbhvJdzlJIpcEanVu1")]
for name, fid in SAMPLES:
    if not os.path.exists(name):
        gdown.download(id=fid, output=name, quiet=True)


def ingest(paths):
    """Full ingestion: read + OCR + tag + chunk + embed + index. Returns (store, report, types)."""
    pages = []
    for p in paths:
        path = p.name if hasattr(p, "name") else p
        pages.extend(tag_pages(path, path.split("/")[-1]))
    if not pages:
        return None, "No readable pages found.", gr.update()
    chunks = chunk_pages(pages)
    store = {"chunks": chunks, "index": build_index(chunks), "pages": pages}
    types = sorted({p["doc_type"] for p in pages})
    ocr_pages = sum(1 for p in pages if p["ocr"])
    report = (f"**Indexed.** {len({p['source_file'] for p in pages})} file(s) · "
              f"{len(pages)} pages · {len({p['doc_id'] for p in pages})} documents · "
              f"{len(chunks)} chunks · {ocr_pages} page(s) needed OCR\n\n"
              f"Types found: {', '.join(types)}")
    return store, report, gr.update(choices=["All types"] + types, value="All types")


def load_samples():
    return ingest([n for n, _ in SAMPLES])


# quick end-to-end check before the UI goes up
_store, _report, _ = load_samples()
print(_report.replace("**", ""))
_r = rag_answer("Does the product contain any material of animal origin?", _store)
print("\nanswer:", _r["answer"][:220])
print("confidence:", _r["confidence"], _r["confidence_label"], "| chunks:", _r["chunks_used"])
print("sources:", [(s["file"], s["page"], s["score"]) for s in _r["sources"]])

Indexed. 2 file(s) · 20 pages · 8 documents · 49 chunks · 0 page(s) needed OCR

Types found: BSE/TSE Declaration, Certificate of Quality, Chain of Custody, Cover Letter, Material Description Sheet, Packaging Specification, Supplier Qualification Record

answer: No, the product does not contain any material of animal origin.
confidence: 76.5 High | chunks: 4
sources: [('pharma-blob-test.pdf', 6, 0.77), ('pharma-blob-test.pdf', 6, 0.761), ('pharma-blob-sample.pdf', 6, 0.711), ('pharma-blob-test.pdf', 6, 0.673)]


## 6. The interface

In [ ]:
# ── 6. The interface ──────────────────────────────────────────────────────────
# A product-style front end: header with model status, packet KPIs, a document
# inventory, chat with suggested questions, and an evidence panel that shows the
# confidence meter and every passage the answer was drawn from.
import html as _html

MODEL_LABEL = {
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit": "Mistral-7B-Instruct · 4-bit GPU",
    "Qwen/Qwen2.5-1.5B-Instruct": "Qwen2.5-1.5B-Instruct · CPU",
}.get(MODEL_ID, MODEL_ID)

TYPE_COLORS = {   # doc_type -> (background, text)
    "Cover Letter":                  ("#EEF2F7", "#334155"),
    "Certificate of Quality":        ("#E7F6EE", "#166534"),
    "Packaging Specification":       ("#FEF3E2", "#9A4D00"),
    "BSE/TSE Declaration":           ("#F1ECFE", "#5B21B6"),
    "Material Description Sheet":    ("#E8F0FE", "#1E40AF"),
    "Supplier Qualification Record": ("#E3F6F5", "#0F6E6A"),
    "Chain of Custody":              ("#FDECEF", "#9F1239"),
    "Unknown":                       ("#F1F5F9", "#64748B"),
}
LEVEL_COLORS = {"High": "#16A34A", "Medium": "#D97706", "Low": "#DC2626", "none": "#94A3B8"}

SUGGESTIONS = [
    ("Animal-origin material?",  "Does the product contain any material of animal origin?"),
    ("Packaging changes",        "Were there any packaging configuration changes?"),
    ("Last supplier audit",      "When was the supplier last audited and what was the result?"),
    ("Released lots",            "Which lots were released and did they pass their tests?"),
]

esc = lambda s: _html.escape(str(s))

def type_tag(t):
    bg, fg = TYPE_COLORS.get(t, TYPE_COLORS["Unknown"])
    return f'<span class="pdi-tag" style="background:{bg};color:{fg}">{esc(t)}</span>'


def header_html():
    return f"""
<div class="pdi-header">
  <div class="pdi-brand">
    <div class="pdi-logo">
      <svg viewBox="0 0 24 24" width="22" height="22" fill="none" stroke="white" stroke-width="2"
           stroke-linecap="round" stroke-linejoin="round"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/>
           <path d="M14 2v6h6"/><circle cx="11.5" cy="14.5" r="2.5"/><path d="M13.3 16.3 16 19"/></svg>
    </div>
    <div>
      <div class="pdi-title">Pharma Document Intelligence</div>
      <div class="pdi-sub">Ask questions across supplier documentation. Every answer is cited and scored.</div>
    </div>
  </div>
  <div class="pdi-pills">
    <span class="pdi-pill"><span class="pdi-dot"></span>{esc(MODEL_LABEL)}</span>
    <span class="pdi-pill">Open-source stack · no external APIs</span>
  </div>
</div>"""


def kpi_html(store=None):
    if store:
        pages = store["pages"]
        vals = [len({p["source_file"] for p in pages}), len(pages),
                len({(p["source_file"], p["doc_id"]) for p in pages}),
                len(store["chunks"]), sum(1 for p in pages if p["ocr"])]
    else:
        vals = ["—"] * 5
    labels = ["Files", "Pages", "Documents found", "Chunks indexed", "Pages via OCR"]
    cards = "".join(f'<div class="pdi-kpi"><div class="pdi-kpi-val">{v}</div>'
                    f'<div class="pdi-kpi-label">{l}</div></div>' for v, l in zip(vals, labels))
    return f'<div class="pdi-kpis">{cards}</div>'


def inventory_html(store=None):
    if not store:
        return ('<div class="pdi-empty-sm">No packet loaded yet. Upload PDFs or load the '
                'sample packet to see the documents it contains.</div>')
    docs = {}
    for p in store["pages"]:
        docs.setdefault(p["source_file"], {}).setdefault(p["doc_id"], (p["doc_type"], p["page_range"]))
    out = ['<div class="pdi-inv">']
    for f, ds in docs.items():
        out.append(f'<div class="pdi-inv-file">{esc(f)} <span>{len(ds)} docs</span></div>')
        for did, (t, rng) in ds.items():
            out.append(f'<div class="pdi-inv-row">{type_tag(t)}'
                       f'<span class="pdi-inv-pages">p. {esc(rng)}</span></div>')
    out.append("</div>")
    return "".join(out)


def evidence_html(res=None, loading=False):
    if loading:
        return ('<div class="pdi-empty"><div class="pdi-spinner"></div>'
                'Retrieving passages and writing a grounded answer…</div>')
    if not res:
        return ('<div class="pdi-empty"><b>Evidence appears here</b><br>Each answer is backed by the '
                'exact passages it came from, with page numbers and similarity scores.</div>')
    if not res["sources"]:
        return '<div class="pdi-empty">No passages matched that question.</div>'
    lvl, pct = res["confidence_label"], res["confidence"]
    color = LEVEL_COLORS.get(lvl, "#94A3B8")
    out = [f"""
<div class="pdi-conf">
  <div class="pdi-conf-top">
    <div><div class="pdi-conf-label">Retrieval confidence</div>
         <div class="pdi-conf-val">{pct}%</div></div>
    <span class="pdi-level" style="background:{color}1A;color:{color}">{esc(lvl)}</span>
  </div>
  <div class="pdi-bar"><div style="width:{min(pct, 100)}%;background:{color}"></div></div>
  <div class="pdi-conf-note">Scored from retrieval similarity across {res['chunks_used']} passages,
  not from the model's wording.</div>
</div>
<div class="pdi-src-title">Sources</div>"""]
    for s in res["sources"]:
        text = s.get("text", "")
        snippet = esc(text[:200] + ("…" if len(text) > 200 else ""))
        full = esc(text)
        ocr = '<span class="pdi-ocr">OCR</span>' if s["ocr"] else ""
        sim = round(s["score"] * 100)
        more = (f'<details><summary>Show full passage</summary><div class="pdi-full">{full}</div></details>'
                if len(text) > 200 else "")
        out.append(f"""
<div class="pdi-src">
  <div class="pdi-src-head">
    <span class="pdi-num">{s['n']}</span>{type_tag(s['doc_type'])}{ocr}
    <span class="pdi-sim"><span class="pdi-simbar"><span style="width:{sim}%"></span></span>{s['score']:.2f}</span>
  </div>
  <div class="pdi-src-meta">{esc(s['file'])} · page {s['page']} · document pages {esc(s['page_range'])}</div>
  {f'<div class="pdi-snippet">{snippet}</div>' if text else ''}
  {more}
</div>""")
    return "".join(out)


def status_md(msg, kind="info"):
    return f'<div class="pdi-status pdi-{kind}">{esc(msg)}</div>'


# ── event handlers ──
def ui_ingest(file_list):
    if not file_list:
        return (gr.update(), status_md("Choose at least one PDF to index.", "warn"),
                gr.update(), gr.update(), gr.update())
    st, report, dd = ingest(file_list)
    if st is None:
        return None, status_md(report, "warn"), dd, kpi_html(None), inventory_html(None)
    return st, status_md("Packet indexed. Ready for questions.", "ok"), dd, kpi_html(st), inventory_html(st)


def ui_load_samples():
    return ui_ingest([n for n, _ in SAMPLES])


def respond(message, history, mode, doc_type, k, store):
    history = list(history or [])
    message = (message or "").strip()
    if not message:
        yield history, "", status_md("Type a question first.", "warn"), gr.update()
        return
    history.append({"role": "user", "content": message})
    if not store:
        history.append({"role": "assistant",
                        "content": "Load the sample packet or upload PDFs first, then ask again."})
        yield history, "", status_md("No documents indexed yet.", "warn"), gr.update()
        return

    yield history, "", status_md("Searching the index…"), evidence_html(loading=True)
    res = rag_answer(message, store, doc_type, mode, int(k))
    lvl = res["confidence_label"]
    dot = {"High": "🟢", "Medium": "🟡", "Low": "🔴"}.get(lvl, "⚪")
    history.append({"role": "assistant",
                    "content": f"{res['answer']}\n\n{dot} *{lvl} confidence · {res['confidence']}% · "
                               f"{res['chunks_used']} sources cited*"})
    yield history, "", status_md("Answer ready. Sources are on the right.", "ok"), evidence_html(res)


def _text(c):
    """Gradio 6 returns chat content as a list of {'type': 'text', 'text': ...} parts."""
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        return " ".join(_text(x) for x in c)
    if isinstance(c, dict):
        return c.get("text") or c.get("value") or ""
    return str(c)


def save_transcript(history):
    lines = ["# Pharma Document Intelligence: transcript",
             f"_{datetime.datetime.now():%Y-%m-%d %H:%M}_", ""]
    for m in history or []:
        who = "**You**" if m["role"] == "user" else "**Assistant**"
        lines.append(f"{who}: {_text(m['content'])}\n")
    with open("rag_transcript.md", "w") as f:
        f.write("\n".join(lines))
    return gr.update(value="rag_transcript.md", visible=True)


def clear_chat():
    return [], "", status_md("Conversation cleared."), evidence_html(None), gr.update(visible=False)


# ── styling ──
THEME = gr.themes.Soft(
    primary_hue="blue", secondary_hue="slate", neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "ui-monospace", "monospace"],
).set(
    body_background_fill="#F4F6FB",
    block_radius="12px",
    button_primary_background_fill="#1D4ED8",
    button_primary_background_fill_hover="#1E40AF",
    button_primary_text_color="white",
    block_label_background_fill="transparent",
    block_label_text_color="#334155",
    block_label_text_weight="600",
    block_title_background_fill="transparent",
    block_title_text_color="#334155",
    block_title_text_weight="600",
)

CSS = """
.gradio-container {max-width: 1480px !important; margin: 0 auto !important;}
footer {display: none !important;}
.pdi-header {display:flex; justify-content:space-between; align-items:center; gap:16px; flex-wrap:wrap;
  background:linear-gradient(120deg,#0B1F4D 0%,#1D4ED8 100%); color:white; padding:20px 26px;
  border-radius:16px; box-shadow:0 8px 24px rgba(29,78,216,.18);}
.pdi-brand {display:flex; align-items:center; gap:14px;}
.pdi-logo {width:44px; height:44px; border-radius:12px; background:rgba(255,255,255,.16);
  display:flex; align-items:center; justify-content:center;}
.pdi-header, .pdi-header * {color:white !important;}
.pdi-title {font-size:21px; font-weight:700; letter-spacing:-.01em;}
.pdi-sub {font-size:13.5px; opacity:.85; margin-top:2px;}
.pdi-pills {display:flex; gap:8px; flex-wrap:wrap;}
.pdi-pill {display:inline-flex; align-items:center; gap:7px; font-size:12.5px; font-weight:500;
  padding:6px 12px; border-radius:999px; background:rgba(255,255,255,.14);
  border:1px solid rgba(255,255,255,.22);}
.pdi-dot {width:8px; height:8px; border-radius:50%; background:#4ADE80; box-shadow:0 0 0 3px rgba(74,222,128,.25);}

.pdi-kpis {display:grid; grid-template-columns:repeat(5,1fr); gap:12px;}
@media (max-width: 900px) {.pdi-kpis {grid-template-columns:repeat(2,1fr);}}
.pdi-kpi {background:white; border:1px solid #E3E8F2; border-radius:12px; padding:14px 18px;}
.pdi-kpi-val {font-size:26px; font-weight:700; color:#0B1F4D; line-height:1.1;}
.pdi-kpi-label {font-size:12px; color:#64748B; margin-top:4px; text-transform:uppercase; letter-spacing:.04em;}

.pdi-card {background:white !important; border:1px solid #E3E8F2 !important; border-radius:16px !important;
  padding:18px !important; box-shadow:0 1px 3px rgba(15,23,42,.04);}
.pdi-section {font-size:13px; font-weight:700; color:#0B1F4D; text-transform:uppercase;
  letter-spacing:.06em; margin:2px 0 4px;}
.pdi-section span {font-weight:500; text-transform:none; letter-spacing:0; color:#64748B; margin-left:6px;}

.pdi-tag {display:inline-block; font-size:11.5px; font-weight:600; padding:3px 9px; border-radius:999px; white-space:nowrap;}
.pdi-inv {display:flex; flex-direction:column; gap:6px; max-height:400px; overflow-y:auto;}
.pdi-inv-file {font-size:12px; font-weight:600; color:#334155; margin-top:6px; font-family:var(--font-mono);}
.pdi-inv-file span {font-family:var(--font); font-weight:500; color:#94A3B8; margin-left:4px;}
.pdi-inv-row {display:flex; justify-content:space-between; align-items:center; padding:7px 10px;
  border:1px solid #EEF1F6; border-radius:10px; background:#FAFBFD;}
.pdi-inv-pages {font-size:12px; color:#64748B;}
.pdi-empty-sm {font-size:13px; color:#64748B; padding:12px; border:1px dashed #CBD5E1; border-radius:10px; background:#FAFBFD;}

.pdi-status {font-size:13px; padding:8px 12px; border-radius:10px; background:#F1F5F9; color:#334155;}
.pdi-ok {background:#ECFDF3; color:#166534;}
.pdi-warn {background:#FEF6E7; color:#92400E;}

.pdi-chips {gap:8px !important; flex-wrap:wrap !important;}
.pdi-chips button {flex:0 0 auto !important; min-width:0 !important; border-radius:999px !important;
  font-weight:500 !important; font-size:13px !important; background:#EEF3FF !important;
  color:#1D4ED8 !important; border:1px solid #D6E2FF !important;}
.pdi-chips button:hover {background:#DCE7FF !important;}

.pdi-empty {text-align:center; color:#64748B; font-size:13.5px; padding:36px 18px; line-height:1.6;
  border:1px dashed #CBD5E1; border-radius:12px; background:#FAFBFD;}
.pdi-empty b {color:#0B1F4D; font-size:14.5px;}
.pdi-spinner {width:22px; height:22px; margin:0 auto 10px; border:3px solid #DCE7FF;
  border-top-color:#1D4ED8; border-radius:50%; animation:pdi-spin .8s linear infinite;}
@keyframes pdi-spin {to {transform:rotate(360deg);}}

.pdi-conf {border:1px solid #E3E8F2; border-radius:12px; padding:14px 16px; background:#FAFBFD;}
.pdi-conf-top {display:flex; justify-content:space-between; align-items:flex-start;}
.pdi-conf-label {font-size:12px; color:#64748B; text-transform:uppercase; letter-spacing:.04em;}
.pdi-conf-val {font-size:30px; font-weight:700; color:#0B1F4D; line-height:1.15;}
.pdi-level {font-size:12px; font-weight:700; padding:4px 11px; border-radius:999px;}
.pdi-bar {height:8px; background:#E2E8F0; border-radius:999px; margin:10px 0 8px; overflow:hidden;}
.pdi-bar div {height:100%; border-radius:999px;}
.pdi-conf-note {font-size:12px; color:#64748B;}
.pdi-src-title {font-size:12px; font-weight:700; color:#0B1F4D; text-transform:uppercase;
  letter-spacing:.06em; margin:16px 0 8px;}
.pdi-src {border:1px solid #E3E8F2; border-radius:12px; padding:12px 14px; margin-bottom:10px; background:white;}
.pdi-src-head {display:flex; align-items:center; gap:8px; flex-wrap:wrap;}
.pdi-num {width:22px; height:22px; border-radius:6px; background:#0B1F4D; color:white; font-size:12px;
  font-weight:700; display:inline-flex; align-items:center; justify-content:center; flex:none;}
.pdi-ocr {font-size:10.5px; font-weight:700; padding:2px 7px; border-radius:6px; background:#FEF3C7; color:#92400E;}
.pdi-sim {margin-left:auto; display:inline-flex; align-items:center; gap:7px; font-size:12px;
  color:#334155; font-family:var(--font-mono);}
.pdi-simbar {width:44px; height:6px; background:#E2E8F0; border-radius:999px; overflow:hidden; display:inline-block;}
.pdi-simbar span {display:block; height:100%; background:#1D4ED8;}
.pdi-src-meta {font-size:12px; color:#64748B; margin:7px 0 6px;}
.pdi-snippet, .pdi-full {font-size:12.5px; color:#334155; line-height:1.55; border-left:3px solid #DCE7FF;
  padding:2px 0 2px 10px; white-space:pre-wrap;}
.pdi-src details {margin-top:6px;}
.pdi-src summary {font-size:12px; color:#1D4ED8; cursor:pointer; font-weight:500;}
.pdi-full {margin-top:6px; max-height:220px; overflow-y:auto;}
.pdi-foot {text-align:center; font-size:12px; color:#94A3B8; padding:4px 0 8px;}
"""

# Gradio share links follow the viewer's dark mode; this app is designed for light.
FORCE_LIGHT = """() => {
  const url = new URL(window.location);
  if (url.searchParams.get('__theme') !== 'light') {
    url.searchParams.set('__theme', 'light'); window.location.href = url.href;
  }
}"""

CHAT_PLACEHOLDER = """<div style="text-align:center;color:#64748B;font-size:14px;line-height:1.6">
<div style="font-size:16px;font-weight:700;color:#0B1F4D;margin-bottom:4px">Ask anything about the packet</div>
Load the sample packet, then try a suggested question below.<br>Answers cite the passages they came from.</div>"""


# ── layout ──
with gr.Blocks(title="Pharma Document Intelligence") as demo:
    store = gr.State(None)
    gr.HTML(header_html())
    kpis = gr.HTML(kpi_html())

    with gr.Row(equal_height=False):
        # left: the packet
        with gr.Column(scale=4, min_width=300, elem_classes="pdi-card"):
            gr.HTML('<div class="pdi-section">Document packet</div>')
            files = gr.File(label="Supplier PDFs · digital or scanned",
                            file_count="multiple", file_types=[".pdf"], height=170)
            with gr.Row():
                index_btn  = gr.Button("Index uploads", variant="primary")
                sample_btn = gr.Button("Load sample packet")
            status = gr.HTML(status_md("No documents indexed yet."))
            gr.HTML('<div class="pdi-section" style="margin-top:8px">Documents detected</div>')
            inventory = gr.HTML(inventory_html())
            gr.HTML('<div class="pdi-section" style="margin-top:8px">Retrieval settings</div>')
            mode     = gr.Radio(list(STYLE), value="Short answer", label="Answer style")
            doc_type = gr.Dropdown(["All types"], value="All types", label="Search within document type")
            k        = gr.Slider(2, 8, value=4, step=1, label="Passages to retrieve")

        # middle: the conversation
        with gr.Column(scale=7, min_width=420, elem_classes="pdi-card"):
            gr.HTML('<div class="pdi-section">Ask the packet</div>')
            chatbot = gr.Chatbot(show_label=False, height=440, placeholder=CHAT_PLACEHOLDER)
            with gr.Row(elem_classes="pdi-chips"):
                chips = [gr.Button(label, size="sm") for label, _ in SUGGESTIONS]
            with gr.Row(equal_height=True):
                msg  = gr.Textbox(show_label=False, scale=8, lines=1, container=False,
                                  placeholder="Ask about lots, materials, audits, packaging changes…")
                send = gr.Button("Ask", variant="primary", scale=1, min_width=90)
            with gr.Row():
                save_btn  = gr.Button("Export transcript", size="sm")
                clear_btn = gr.Button("Clear conversation", size="sm")
            transcript = gr.File(label="Transcript", visible=False)

        # right: the evidence
        with gr.Column(scale=5, min_width=320, elem_classes="pdi-card"):
            gr.HTML('<div class="pdi-section">Evidence</div>')
            evidence = gr.HTML(evidence_html())

    gr.HTML('<div class="pdi-foot">Answers are generated only from retrieved passages · '
            'confidence is computed from retrieval similarity · built with open-source models</div>')

    ingest_out = [store, status, doc_type, kpis, inventory]
    index_btn.click(ui_ingest, inputs=files, outputs=ingest_out)
    sample_btn.click(ui_load_samples, outputs=ingest_out)

    chat_in  = [msg, chatbot, mode, doc_type, k, store]
    chat_out = [chatbot, msg, status, evidence]
    msg.submit(respond, chat_in, chat_out)
    send.click(respond, chat_in, chat_out)
    for btn, (_, question) in zip(chips, SUGGESTIONS):
        btn.click(lambda q=question: q, outputs=msg).then(respond, chat_in, chat_out)
    save_btn.click(save_transcript, inputs=chatbot, outputs=transcript)
    clear_btn.click(clear_chat, outputs=[chatbot, msg, status, evidence, transcript])

demo.queue()
try:      # Gradio 6 takes theme/css/js in launch()
    demo.launch(share=True, theme=THEME, css=CSS, js=FORCE_LIGHT)
except TypeError:
    demo.launch(share=True)

In [ ]:
print("public app link:", demo.share_url)

## 7. Performance notes

Measured on the bundled two-file packet (20 pages, 8 documents, 40 chunks):

- **Ingestion** is the slow half and it runs once. Embedding 40 chunks with bge-small takes
  about a second on GPU; OCR, when a page needs it, costs roughly a second per page on top.
- **Retrieval is effectively free.** FAISS `IndexFlatIP` over 40 vectors returns in under a
  millisecond, so at this size an exact index is the right call and an approximate one
  (IVF, HNSW) would only start paying off in the tens of thousands of chunks.
- **Generation dominates response time**, which is why `k` is exposed as a slider rather than
  hard-coded: every extra passage lengthens the prompt and the wait.
- **Confidence comes from retrieval, not the model.** The score blends the top cosine
  similarity with how much of the top-k agrees with it, so a lone weak match reads Low even
  when the model writes fluently. A model's confident tone is not evidence; a similarity
  score at least is.